In [1]:
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = Path("../Outputs")
master = pd.read_csv(OUTPUTS_DIR / "routed_master_905.csv")
llm_emo_obj = pd.read_csv(OUTPUTS_DIR / "llm_emotion_objective_905.csv")

LOCKED_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

# Merge LLM scores into master via _post_index
llm_renamed = llm_emo_obj.rename(columns={e: f"llm_{e}" for e in LOCKED_EMOTIONS})
merged = master.merge(
    llm_renamed[[f"llm_{e}" for e in LOCKED_EMOTIONS] + ["_post_index"]],
    left_index=True,
    right_on="_post_index",
    how="inner"
)

print(f"Yellow posts with LLM scores: {len(merged)}\n")

# Get dominant emotion per source
def dominant(row, prefix):
    scores = {e: row[f"{prefix}_{e}"] for e in LOCKED_EMOTIONS}
    return max(scores, key=scores.get)

merged["distil_dom"] = merged.apply(lambda r: dominant(r, "distil"), axis=1)
merged["goemo_dom"] = merged.apply(lambda r: dominant(r, "goemo"), axis=1)
merged["cardiff_dom"] = merged.apply(lambda r: dominant(r, "cardiff"), axis=1)
merged["llm_dom"] = merged.apply(lambda r: dominant(r, "llm"), axis=1)

# Pairwise agreement rates
print("=== Pairwise dominant emotion agreement ===")
pairs = [
    ("DistilRoBERTa", "GoEmotions", "distil_dom", "goemo_dom"),
    ("DistilRoBERTa", "Cardiff", "distil_dom", "cardiff_dom"),
    ("GoEmotions", "Cardiff", "goemo_dom", "cardiff_dom"),
    ("DistilRoBERTa", "LLM", "distil_dom", "llm_dom"),
    ("GoEmotions", "LLM", "goemo_dom", "llm_dom"),
    ("Cardiff", "LLM", "cardiff_dom", "llm_dom"),
]
for name1, name2, col1, col2 in pairs:
    agree = (merged[col1] == merged[col2]).sum()
    pct = agree / len(merged) * 100
    print(f"  {name1:15s} vs {name2:15s}: {agree:4d}/{len(merged)} ({pct:.1f}%)")

# Critical question: when Cardiff disagrees with the others, who does LLM side with?
print("\n=== Diagnostic: when Cardiff is the outlier, who does LLM agree with? ===")
cardiff_outlier = merged[
    (merged["distil_dom"] == merged["goemo_dom"]) &  # other two agree
    (merged["cardiff_dom"] != merged["distil_dom"])  # cardiff disagrees
]
print(f"Posts where Cardiff disagrees with both others: {len(cardiff_outlier)}")
if len(cardiff_outlier) > 0:
    llm_sided_with_cardiff = (cardiff_outlier["llm_dom"] == cardiff_outlier["cardiff_dom"]).sum()
    llm_sided_with_others = (cardiff_outlier["llm_dom"] == cardiff_outlier["distil_dom"]).sum()
    llm_picked_third = len(cardiff_outlier) - llm_sided_with_cardiff - llm_sided_with_others
    print(f"  LLM sided with Cardiff: {llm_sided_with_cardiff} ({llm_sided_with_cardiff/len(cardiff_outlier)*100:.1f}%)")
    print(f"  LLM sided with others (Distil + GoEmo): {llm_sided_with_others} ({llm_sided_with_others/len(cardiff_outlier)*100:.1f}%)")
    print(f"  LLM picked something different from all: {llm_picked_third} ({llm_picked_third/len(cardiff_outlier)*100:.1f}%)")

Yellow posts with LLM scores: 696

=== Pairwise dominant emotion agreement ===
  DistilRoBERTa   vs GoEmotions     :  246/696 (35.3%)
  DistilRoBERTa   vs Cardiff        :  289/696 (41.5%)
  GoEmotions      vs Cardiff        :  288/696 (41.4%)
  DistilRoBERTa   vs LLM            :  255/696 (36.6%)
  GoEmotions      vs LLM            :  292/696 (42.0%)
  Cardiff         vs LLM            :  474/696 (68.1%)

=== Diagnostic: when Cardiff is the outlier, who does LLM agree with? ===
Posts where Cardiff disagrees with both others: 92
  LLM sided with Cardiff: 47 (51.1%)
  LLM sided with others (Distil + GoEmo): 13 (14.1%)
  LLM picked something different from all: 32 (34.8%)
